# Section 3: Cross-Generator Evaluation

Both detectors trained in `02_train_models.ipynb` saw FaceSwap fakes
during training but **not** Stable Diffusion. This notebook measures
the resulting transfer gap.

Test sets:

| CSV               | Composition                              | What it measures        |
|-------------------|-------------------------------------------|--------------------------|
| `test_indist.csv` | 400 FFHQ + 400 FaceSwap                   | In-distribution accuracy |
| `test_ood.csv`    | 400 FFHQ + 1000 Stable Diffusion v1.5     | Cross-generator transfer |

We also break down `test_ood` by source to separate "holds up on real
images" from "actually catches diffusion fakes."

## 0. Colab Setup

Three cells, idempotent: clone repo if missing, mount Drive, restore
the test split CSVs and checkpoints from Drive.

In [ ]:
import os
from pathlib import Path

REPO_URL  = "https://github.com/phanindra-max/Cross-Generator-Generalization-Project.git"
REPO_NAME = "Cross-Generator-Generalization-Project"

if not Path("src").exists() and not Path(REPO_NAME).exists():
    !git clone -q {REPO_URL}

if Path(REPO_NAME, "src").exists() and not Path("src").exists():
    %cd {REPO_NAME}

!pip install -q uv
!uv pip install -q -r requirements.txt

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
assert (PROJECT_ROOT / "src").exists(), f"src/ not found under {PROJECT_ROOT}"
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Working dir: {PROJECT_ROOT}")

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_BACKUP = Path("/content/drive/MyDrive/deepfake-cross-generator/backups")
    IN_COLAB = True
    print(f"Drive backup root: {DRIVE_BACKUP}")
except ImportError:
    IN_COLAB = False
    DRIVE_BACKUP = None
    print("Not in Colab \u2014 Drive mount skipped.")

In [ ]:
# Restore test data + trained checkpoints from Drive into the runtime,
# only if local target is empty.
# Note: a directory containing only dotfiles (e.g. .gitkeep, committed in the
# repo as a placeholder) is treated as empty so the restore proceeds.
import subprocess

ARTIFACTS = [
    ("data/real",           "real.tar"),
    ("data/faceswap",       "faceswap.tar"),
    ("data/diffusion_sd15", "diffusion_sd15.tar"),
    ("data/splits",         "splits.tar"),
    ("results/checkpoints", "checkpoints.tar"),
    ("results/metrics",     "metrics.tar"),
]

def _is_empty(p: Path) -> bool:
    if not p.exists():
        return True
    return not any(not child.name.startswith(".") for child in p.iterdir())

if IN_COLAB and DRIVE_BACKUP and DRIVE_BACKUP.exists():
    for local, archive in ARTIFACTS:
        target = Path(local)
        src = DRIVE_BACKUP / archive
        if not _is_empty(target):
            print(f"  skip {local}: already populated")
            continue
        if not src.exists():
            print(f"  miss {local}: no archive in Drive yet")
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["tar", "-xf", str(src), "-C", str(target.parent)], check=True)
        print(f"  ok   {local} restored from {archive}")
else:
    print("No Drive backup root — skipping restore.")

# If splits weren't backed up, regenerate them on the fly.
if not Path("data/splits/test_indist.csv").exists() or not Path("data/splits/test_ood.csv").exists():
    print("\nSplit CSVs missing — regenerating from data/{real,faceswap,diffusion_sd15}/...")
    from src.data.create_splits import create_splits
    create_splits(
        real_dir="data/real",
        faceswap_dir="data/faceswap",
        diffusion_dir="data/diffusion_sd15",
        output_dir="data/splits",
        seed=42,
    )

for required in ("data/splits/test_indist.csv", "data/splits/test_ood.csv"):
    assert Path(required).exists(), f"{required} missing — run 00_data_pipeline first."
for ckpt in ("results/checkpoints/constrained_cnn_best.pth", "results/checkpoints/baseline_resnet_best.pth"):
    if not Path(ckpt).exists():
        print(f"  WARN: missing {ckpt} — run 02_train_models or restore checkpoints.tar")

## 3.0 Imports

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader

from src.data.faceforensics_loader import FaceForensicsDataset
from src.models.constrained_cnn import ConstrainedCNN
from src.models.baseline_resnet import BaselineResNet
from src.evaluation.stratified_metrics import (
    compute_metrics,
    cross_generator_evaluation,
    print_cross_generator_table,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

%matplotlib inline
try:
    plt.style.use("seaborn-v0_8-paper")
except OSError:
    plt.style.use("default")
plt.rcParams["figure.dpi"] = 120

## 3.1 Paths and split summary

In [ ]:
SPLITS_DIR     = Path("data/splits")
CHECKPOINT_DIR = Path("results/checkpoints")
METRICS_DIR    = Path("results/metrics")
FIGURES_DIR    = Path("results/figures")
for d in (METRICS_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

for csv in ("test_indist.csv", "test_ood.csv"):
    df = pd.read_csv(SPLITS_DIR / csv)
    print(f"{csv:<18} {len(df):>5} rows  sources: {df['source'].value_counts().to_dict()}")

## 3.2 Load trained checkpoints

In [ ]:
def load_checkpoint(model_name: str, model: torch.nn.Module) -> torch.nn.Module:
    ckpt_path = CHECKPOINT_DIR / f"{model_name}_best.pth"
    assert ckpt_path.exists(), (
        f"Missing {ckpt_path}. Train it in 02_train_models.ipynb (or restore "
        f"results/checkpoints from Drive)."
    )
    state = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(state["model_state_dict"])
    model = model.to(DEVICE).eval()
    print(f"  {model_name:<18} epoch={state['epoch']}  best_val_auc={state['best_auc']:.4f}")
    return model

constrained = load_checkpoint("constrained_cnn",  ConstrainedCNN(num_classes=2))
baseline    = load_checkpoint("baseline_resnet", BaselineResNet(num_classes=2, backbone="resnet50", pretrained=False))

## 3.3 Test DataLoaders

Eval-mode transforms only \u2014 no augmentation. ImageNet normalization to
match training.

In [ ]:
RES = 128
BS  = 64
NW  = 2

eval_tf = A.Compose([
    A.Resize(RES, RES),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

def loader_for_csv(csv_path: str) -> DataLoader:
    ds = FaceForensicsDataset(csv_path=csv_path, transform=eval_tf)
    return DataLoader(ds, batch_size=BS, shuffle=False, num_workers=NW,
                       pin_memory=(DEVICE.type == "cuda"))

test_loaders = {
    "in_dist (FaceSwap)":         loader_for_csv(str(SPLITS_DIR / "test_indist.csv")),
    "ood (Stable Diffusion)":     loader_for_csv(str(SPLITS_DIR / "test_ood.csv")),
}
for name, loader in test_loaders.items():
    print(f"  {name:<28} {len(loader.dataset)} samples / {len(loader)} batches")

## 3.4 Run evaluation on both models

In [ ]:
results_constrained = cross_generator_evaluation(constrained, test_loaders, str(DEVICE))
results_baseline    = cross_generator_evaluation(baseline,    test_loaders, str(DEVICE))

print("=== ConstrainedCNN ===")
print(print_cross_generator_table(results_constrained))
print("\n=== BaselineResNet ===")
print(print_cross_generator_table(results_baseline))

## 3.5 Headline plot: AUC by test set

Two models, two test sets. The gap between the in-dist and OOD bars is
the cross-generator transfer cost.

In [ ]:
test_names = list(test_loaders.keys())
constrained_aucs = [results_constrained[n]["auc"] for n in test_names]
baseline_aucs    = [results_baseline[n]["auc"]    for n in test_names]

fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(test_names))
w = 0.35
ax.bar(x - w/2, constrained_aucs, w, label="ConstrainedCNN", color="#2196F3")
ax.bar(x + w/2, baseline_aucs,    w, label="BaselineResNet", color="#FF9800")
ax.axhline(0.5, color="red", linestyle="--", alpha=0.5, label="random")
ax.set_xticks(x)
ax.set_xticklabels(test_names)
ax.set_ylabel("AUC")
ax.set_ylim(0.4, 1.05)
ax.set_title("Cross-Generator Transfer: in-dist vs. unseen Stable Diffusion")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "cross_generator_auc.png", dpi=150, bbox_inches="tight")
plt.show()

## 3.6 Save metrics CSV

In [ ]:
rows = []
for test_name in test_names:
    rc = results_constrained[test_name]
    rb = results_baseline[test_name]
    rows.append({
        "test_set":           test_name,
        "constrained_acc":    rc["accuracy"], "constrained_auc": rc["auc"],
        "constrained_eer":    rc["eer"],      "constrained_f1":  rc["f1"],
        "baseline_acc":       rb["accuracy"], "baseline_auc":    rb["auc"],
        "baseline_eer":       rb["eer"],      "baseline_f1":     rb["f1"],
        "auc_advantage":      rc["auc"] - rb["auc"],
    })
results_df = pd.DataFrame(rows)
results_df.to_csv(METRICS_DIR / "cross_generator_results.csv", index=False)
results_df

## 3.7 Stratified breakdown of `test_ood` by source

`test_ood.csv` mixes 400 FFHQ-real and 1000 Stable-Diffusion-fake. The
aggregate AUC can hide failure modes \u2014 e.g. a model that calls everything
real will get the FFHQ rows right and the SD rows wrong, with overall
AUC near 0.5.

Below: per-class hit rate (does the model predict the correct label for
this source group?) and mean fake-probability per source.

In [ ]:
@torch.no_grad()
def predict_with_paths(model, csv_path: str):
    """Run inference and return the probabilities + the original CSV (with source)."""
    df = pd.read_csv(csv_path)
    ds = FaceForensicsDataset(csv_path=csv_path, transform=eval_tf)
    loader = DataLoader(ds, batch_size=BS, shuffle=False, num_workers=NW,
                        pin_memory=(DEVICE.type == "cuda"))
    probs = []
    model.eval()
    for images, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        out = model(images)
        probs.append(torch.softmax(out, dim=1)[:, 1].cpu().numpy())
    df = df.copy()
    df["prob_fake"] = np.concatenate(probs)
    df["pred"]      = (df["prob_fake"] >= 0.5).astype(int)
    df["correct"]   = (df["pred"] == df["label"]).astype(int)
    return df

ood_csv = str(SPLITS_DIR / "test_ood.csv")
ood_constrained = predict_with_paths(constrained, ood_csv)
ood_baseline    = predict_with_paths(baseline,    ood_csv)

def per_source_summary(df: pd.DataFrame, model_name: str) -> pd.DataFrame:
    grp = df.groupby("source")
    out = grp.agg(
        n=("label", "size"),
        label=("label", "first"),
        accuracy=("correct", "mean"),
        mean_prob_fake=("prob_fake", "mean"),
    ).reset_index()
    out.insert(0, "model", model_name)
    return out

stratified = pd.concat([
    per_source_summary(ood_constrained, "ConstrainedCNN"),
    per_source_summary(ood_baseline,    "BaselineResNet"),
], ignore_index=True)
stratified.to_csv(METRICS_DIR / "ood_per_source.csv", index=False)
stratified

## 3.8 How to read the results

- **Aggregate AUC drop (in_dist \u2192 ood)** is the headline transfer gap.
  Bigger drop = worse cross-generator generalization.
- **Per-source breakdown in 3.7**: an OOD AUC near 0.5 is most often
  driven by `mean_prob_fake` for the SD rows being too low \u2014 i.e. the
  model misses synthetic fakes outright. If FFHQ rows still get correct
  labels, the model has not collapsed; it just doesn't recognize a new
  generator's fingerprint.
- **ConstrainedCNN vs BaselineResNet**: if `auc_advantage` (in 3.6) is
  positive on `ood (Stable Diffusion)`, the prediction-error front end
  is helping with cross-family transfer \u2014 the headline of the original
  Bayar & Stamm story.

If the OOD AUC sits near random for both models, the next experiment
(per Wang et al. 2020 in `research-notes.md`) is heavy training-time
augmentation \u2014 random JPEG / blur / cropping \u2014 to see if it forces the
models off architecture-specific features.